## Setup

In [ ]:
#Configuration colab

import os
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    PROJECT_ROOT = Path("/content/noo-far-pipeline")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/noofar-ia/noo-far-pipeline.git /content/noo-far-pipeline
    else:
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

import sys
sys.path.insert(0, str(PROJECT_ROOT))
!pip install chromadb rank-bm25 --quiet
print(f"Projet : {PROJECT_ROOT} | Sur Colab : {ON_COLAB}")

In [ ]:
from getpass import getpass
import os
os.environ["HF_TOKEN"] = getpass("HF token : ")
hf_token = os.environ["HF_TOKEN"]

## Verifier que les fichiers wolofs sont présents 

In [ ]:
fiches_wo_dir = PROJECT_ROOT / "rag" / "fiches" / "wo"
fiches = sorted(fiches_wo_dir.glob("*.md"))
print(f"{len(fiches)} fiches wolof trouvées : {[f.name for f in fiches]}")

## Indexer les fiches wolof

In [ ]:
from rag.indexer import build_index

collection = build_index(lang="wo")

In [ ]:
from rag.indexer import build_index
build_index(lang="fr")

In [ ]:
# Verification rapide 
print(collection.count())


In [ ]:
print(QUESTIONS_TEST_WO[-1].keys())

## Test critique 

In [ ]:
# test en mode semantic

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from rag.questions_test_wo import QUESTIONS_TEST_WO
from rag.retriever import retrieve

resultats_test = []
for item in QUESTIONS_TEST_WO:
    passages = retrieve(item["question"], lang="wo", mode="semantic")
    top3_sources = [meta["source"] for doc, meta in passages]
    correct = any(item["intent_attendu"] in source for source in top3_sources)
    resultats_test.append({**item, "top3": top3_sources, "correct_top3": correct})
    print(f"{'OK' if correct else 'ECHEC'} — {item['question']}")

score = sum(r["correct_top3"] for r in resultats_test) / len(resultats_test)
print(f"\nScore top-3 (semantic, traduction verifiee) : {score:.0%}")

In [ ]:
resultats_hybrid = []
for item in QUESTIONS_TEST_WO:
    passages = retrieve(item["question"], lang="wo", mode="hybrid")
    top3_sources = [meta["source"] for doc, meta in passages]
    correct = any(item["intent_attendu"] in source for source in top3_sources)
    resultats_hybrid.append({**item, "top3": top3_sources, "correct_top3": correct})
    print(f"{'OK' if correct else 'ECHEC'} — {item['question']}")
    
score = sum(r["correct_top3"] for r in resultats_hybrid) / len(resultats_hybrid)
print(f"\nScore top-3 (semantic, traduction verifiee) : {score:.0%}")

In [ ]:
#Calculer le score global

import pandas as pd

df_test = pd.DataFrame(resultats_test)
score = df_test["correct_top3"].sum() / len(df_test)
print(f"Score top-3 (semantic) : {score:.0%} ({df_test['correct_top3'].sum()}/{len(df_test)})")

## Analyse et conclusion
*Résultats mesurés :**
- Score top-3 (semantic) : 7/10
- Score top-3 (hybrid) : 9/10

**Décision retenue :** [semantic | hybrid], pour la raison suivante : [...]

**Mise à jour config.yaml :** `retrieval: [valeur]` — [même valeur que le français, ou différenciée par langue si le code le permet ; noter si une évolution du code est nécessaire pour supporter un mode différent par langue]